# QueryLane specialist fine-tune — Unsloth (Colab T4)

Preferred notebook: ~2x faster than the vanilla TRL one, with built-in GGUF export.
If the Unsloth install errors on a fresh Colab image, fall back to `colab_finetune.ipynb`.

**Runtime → Change runtime type → T4 GPU**, then run top to bottom.
Outputs land in Drive under `ft-out/`: `ft-1.5b-q4_k_m.gguf`, `ft-3b-q4_k_m.gguf`.

In [ ]:
!pip install -q unsloth
import unsloth, torch
print('unsloth ready | cuda:', torch.cuda.get_device_name(0))

In [ ]:
# Upload train_all.jsonl (from the repo's finetune/data/) when prompted
from google.colab import files
uploaded = files.upload()
DATA = list(uploaded.keys())[0]
print('rows:', sum(1 for _ in open(DATA)))

In [ ]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

def train_and_export(base, tag, batch, accum, no_thinking=False):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base, max_seq_length=1024, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=7)
    # Qwen3-family templates default to thinking mode; our deployment runs at
    # ~2 tok/s on CPU, so training targets must be direct answers only.
    tmpl = {'enable_thinking': False} if no_thinking else {}
    rows = [json.loads(l) for l in open(DATA) if l.strip()]
    ds = Dataset.from_list([
        {'text': tokenizer.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False, **tmpl)}
        for r in rows
    ])
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds, dataset_text_field='text',
        args=SFTConfig(output_dir=f'/content/ft-{tag}', num_train_epochs=3,
                       learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
                       per_device_train_batch_size=batch, gradient_accumulation_steps=accum,
                       fp16=True, logging_steps=25, save_strategy='no',
                       max_seq_length=1024, packing=False, report_to=[]))
    trainer.train()
    # merge + GGUF + Q4_K_M in one maintained call (builds llama.cpp on first use)
    model.save_pretrained_gguf(f'/content/gguf-{tag}', tokenizer, quantization_method='q4_k_m')
    import glob, shutil, torch as t
    # Unsloth may write into '<dir>' or '<dir>_gguf/' depending on version;
    # search both, any depth, case-insensitive quant suffix.
    src = [p for p in glob.glob(f'/content/gguf-{tag}*/**/*.gguf', recursive=True)
           if 'q4_k_m' in p.lower()]
    assert src, f'no Q4_K_M gguf found under /content/gguf-{tag}*'
    shutil.move(src[0], f'/content/ft-{tag}-q4_k_m.gguf')
    del model, trainer; t.cuda.empty_cache()
    print('exported', f'/content/ft-{tag}-q4_k_m.gguf')

In [ ]:
train_and_export('unsloth/Qwen2.5-1.5B-Instruct', '1.5b', batch=8, accum=2)

In [ ]:
train_and_export('unsloth/Qwen2.5-3B-Instruct', '3b', batch=4, accum=4)

In [ ]:
# OPTIONAL third A/B candidate: newer base, slightly slower/bigger at deployment
# (6.9 tok/s / 1.4GB vs 8.2 tok/s / 1.1GB for Qwen2.5-1.5B in our grading sandbox),
# and zero-shot it lost both probes to Qwen2.5-1.5B. Fine-tuning may flip that.
# Run only if session time allows; the dev judge picks the winner per category.
train_and_export('unsloth/Qwen3.5-2B-Instruct', 'q35-2b', batch=6, accum=3, no_thinking=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ft-out
!cp /content/ft-*-q4_k_m.gguf /content/drive/MyDrive/ft-out/
!ls -lh /content/drive/MyDrive/ft-out/